# TML 1903 — A100 Layout — VPS MODEL
Robust path: Python 3.12 isolated env, GPU Paddle, exact layout stack, model copied from the working VPS cache. No Hugging Face/ModelScope model download.


In [ ]:
import os, shutil, subprocess, sys
REPO='/content/Tennis-OCR-Pipeline'; VENV='/content/tml-layout-py312'; VENV_PY=f'{VENV}/bin/python'
PADDLE_URL='https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'
PADDLE_WHL='/content/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'; EXPECTED=1890365820
print('1/4 repository',flush=True)
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
print('2/4 Python 3.12 isolated environment',flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
UV=shutil.which('uv'); assert UV
subprocess.run([UV,'python','install','3.12'],check=True)
if not os.path.exists(VENV_PY): subprocess.run([UV,'venv','--seed','--python','3.12',VENV],check=True)
print('3/4 Paddle GPU',flush=True)
need_paddle=subprocess.run([VENV_PY,'-c',"import paddle,sys; sys.exit(0 if paddle.__version__=='3.2.0' and paddle.device.is_compiled_with_cuda() else 1)"]).returncode!=0
if need_paddle:
 if not os.path.exists(PADDLE_WHL) or os.path.getsize(PADDLE_WHL)!=EXPECTED:
  subprocess.run(['curl','-L','--fail','--retry','5','-C','-','--progress-bar','-o',PADDLE_WHL,PADDLE_URL],check=True)
 subprocess.run([VENV_PY,'-m','pip','install','--progress-bar','on',PADDLE_WHL],check=True)
else: print('Paddle GPU 3.2.0 already ready — skipping 1.89 GB reinstall',flush=True)
print('4/4 exact layout stack from working VPS',flush=True)
subprocess.run([VENV_PY,'-m','pip','install','--progress-bar','on','paddlex==3.7.2','paddleocr==3.7.0','opencv-contrib-python==4.10.0.84','paramiko>=3.5,<4'],check=True)
subprocess.run([VENV_PY,'-c',"import paddle,paddlex,cv2; print('READY Python/Paddle/PaddleX/OpenCV',paddle.__version__,paddlex.__version__,cv2.__version__,'CUDA=',paddle.device.is_compiled_with_cuda())"],check=True)
print('ENV READY',flush=True)


In [ ]:
from google.colab import files
import base64
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload tml_colab_ed25519')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload private key, not .pub')
KEY_B64=base64.b64encode(data).decode(); print('KEY READY',name,flush=True)


In [ ]:
import subprocess
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'; VPS_USER='andre'; VPS_PORT=2222
CLAIM='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_layout_active_claim.tsv'
STOP='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_layout_quality_complete_1903.flag'
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip(); print('CODE',commit,flush=True)
cmd=[VENV_PY,'-u',f'{REPO}/colab/layout_pool.py','--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',KEY_B64,'--vps-port',str(VPS_PORT),'--claim',CLAIM,'--stop-flag',STOP,'--workers','4','--downloaders','8','--poll','10']
print('STARTING: model will be copied from VPS cache, then 4 GPU workers are preflighted before any claim images are downloaded.',flush=True)
subprocess.run(cmd,check=True)
